In [ ]:
import pandas as pd
import numpy as np
import warnings
import gc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, callbacks
from itertools import combinations

# Boosters
import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

# Sklearn
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1. Configuration
# ---------------------------------------------------------
class Config:
    seed = 2025
    n_folds = 10 
    input_dir = "/kaggle/input/playground-series-s5e12/"
    target = 'diagnosed_diabetes'
    
gpus = tf.config.experimental.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError as e:
        print(e)

warnings.filterwarnings('ignore')

# 2. Feature Engineering & Processing
# ---------------------------------------------------------
def apply_external_encoding(train, test, orig, target_col):
    df_train = train.copy()
    df_test = test.copy()
    
    common_cols = [c for c in train.columns if c in orig.columns and c != target_col and c != 'id']
    print(f"Generating External Encodings for {len(common_cols)} features...")
    
    for col in common_cols:
        mean_map = orig.groupby(col)[target_col].mean()
        df_train[f'ext_mean_{col}'] = df_train[col].map(mean_map)
        df_test[f'ext_mean_{col}'] = df_test[col].map(mean_map)
        
        count_map = orig.groupby(col).size()
        total_orig = len(orig)
        df_train[f'ext_freq_{col}'] = df_train[col].map(count_map) / total_orig
        df_test[f'ext_freq_{col}'] = df_test[col].map(count_map) / total_orig
            
    new_cols = [c for c in df_train.columns if 'ext_' in c]
    for c in new_cols:
        if 'mean' in c:
            global_mean = orig[target_col].mean()
            df_train[c] = df_train[c].fillna(global_mean)
            df_test[c] = df_test[c].fillna(global_mean)
        else:
            df_train[c] = df_train[c].fillna(0)
            df_test[c] = df_test[c].fillna(0)
    return df_train, df_test

def engineer_features(df):
    df = df.copy()
    
    # Medical Ratios
    df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
    df['map'] = df['diastolic_bp'] + (df['pulse_pressure'] / 3)
    df['insulin_resistance'] = df['triglycerides'] / (df['hdl_cholesterol'] + 1e-5)
    df['cholesterol_ratio'] = df['cholesterol_total'] / (df['hdl_cholesterol'] + 1e-5)
    df['non_hdl'] = df['cholesterol_total'] - df['hdl_cholesterol']
    df['visceral_fat'] = df['bmi'] * df['waist_to_hip_ratio']
    df['body_shape_risk'] = df['waist_to_hip_ratio'] * df['triglycerides']
    
    # Risk Scores
    df['metabolic_score'] = (
        (df['bmi'] > 30).astype(int) + 
        (df['waist_to_hip_ratio'] > 0.9).astype(int) +
        (df['systolic_bp'] > 130).astype(int) +
        (df['triglycerides'] > 150).astype(int) + 
        (df['hdl_cholesterol'] < 40).astype(int)
    )
    
    df['age_bmi_interaction'] = df['age'] * df['bmi']
    df['genetic_risk_score'] = df['family_history_diabetes'] * (df['age'] + df['bmi'])
    
    # Lifestyle
    activity_hours = df['physical_activity_minutes_per_week'] / 60
    df['sedentary_ratio'] = df['screen_time_hours_per_day'] / (activity_hours + 1e-5)
    
    # Transforms
    for col in ['triglycerides', 'bmi', 'insulin_resistance']:
        df[f'log_{col}'] = np.log1p(df[col])
    
    for col in ['bmi', 'waist_to_hip_ratio', 'age', 'systolic_bp']:
        df[f'sq_{col}'] = df[col] ** 2
            
    return df

def engineer_categorical_pairs(df, cat_cols):
    df = df.copy()
    pairs = list(combinations(cat_cols, 2))
    print(f"Generating {len(pairs)} categorical interaction features...")
    for col1, col2 in pairs:
        new_col = f"{col1}_{col2}_interact"
        df[new_col] = df[col1].astype(str) + '_' + df[col2].astype(str)
    return df

# 3. Data Prep Pipelines
# ---------------------------------------------------------
def prepare_data_dual(df_train, df_test, cat_cols, num_cols):
    len_train = len(df_train)
    combined = pd.concat([df_train, df_test], axis=0).reset_index(drop=True)
    
    # A. Native Pipeline (Trees) - Float32 + Category
    native = combined.copy()
    for col in num_cols:
        native[col] = native[col].astype(np.float32)
    
    all_cat_cols = native.select_dtypes(include=['object', 'category']).columns.tolist()
    for col in all_cat_cols:
        native[col] = native[col].astype('category')
        
    X_native = native.iloc[:len_train]
    X_test_native = native.iloc[len_train:]
    
    # B. Neural Pipeline - Scaled + OneHot
    print("Preparing Neural Network Data...")
    # Only use original cats for NN to keep dimensionality manageable
    nn_cats = [c for c in cat_cols if c in combined.columns] 
    
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), num_cols),
            ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), nn_cats)
        ],
        verbose_feature_names_out=False
    )
    nn_data = preprocessor.fit_transform(combined)
    X_nn = nn_data[:len_train]
    X_test_nn = nn_data[len_train:]
    
    return X_native, X_test_native, X_nn, X_test_nn

# 4. Neural Network
# ---------------------------------------------------------
def get_nn_model(input_dim):
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(256, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.3),
        layers.Dense(128, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.2),
        layers.Dense(64, activation='swish'),
        layers.BatchNormalization(),
        layers.Dropout(0.1),
        layers.Dense(1, activation='sigmoid')
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=[keras.metrics.AUC(name='auc')]
    )
    return model

# 5. Hill Climbing
# ---------------------------------------------------------
def hill_climbing(oof_df, target, test_preds_df):
    models = [col for col in oof_df.columns if col != target]
    print(f"\n--- Hill Climbing Optimization: {models} ---")
    
    best_weights = {m: 0 for m in models}
    current_preds = np.zeros(len(oof_df))
    iterations = 2000 # Increased iterations for better precision
    current_best_auc = 0
    
    for i in range(iterations):
        best_model = None
        best_iter_auc = 0
        for model in models:
            temp_preds = (current_preds + oof_df[model]) / (i + 1)
            auc = roc_auc_score(oof_df[target], temp_preds)
            if auc > best_iter_auc:
                best_iter_auc = auc
                best_model = model
        current_preds += oof_df[best_model]
        best_weights[best_model] += 1
        current_best_auc = best_iter_auc
    
    final_weights = {m: w/iterations for m, w in best_weights.items()}
    print("Optimized Weights:")
    for m, w in final_weights.items():
        print(f"  {m}: {w:.4f}")
    print(f"Optimized Ensemble OOF AUC: {current_best_auc}")
    
    final_test_pred = np.zeros(len(test_preds_df))
    for m, w in final_weights.items():
        final_test_pred += test_preds_df[m] * w
    return final_test_pred, current_best_auc

# 6. Main Loop
# ---------------------------------------------------------
if __name__ == "__main__":
    print("Loading Data...")
    train_df = pd.read_csv(f"{Config.input_dir}train.csv")
    test_df = pd.read_csv(f"{Config.input_dir}test.csv")
    orig_df = pd.read_csv("/kaggle/input/diabetes-health-indicators-dataset/diabetes_dataset.csv")

    if 'Diabetes_binary' in orig_df.columns:
        orig_df = orig_df.rename(columns={'Diabetes_binary': Config.target})

    # Engineering
    train_df, test_df = apply_external_encoding(train_df, test_df, orig_df, Config.target)
    
    base_cat_cols = ['gender', 'ethnicity', 'employment_status', 'education_level', 'income_level', 'smoking_status']
    train_df = engineer_categorical_pairs(train_df, base_cat_cols)
    test_df = engineer_categorical_pairs(test_df, base_cat_cols)
    
    train_df = engineer_features(train_df)
    test_df = engineer_features(test_df)
    
    y = train_df[Config.target]
    test_ids = test_df['id']
    X = train_df.drop([Config.target, 'id'], axis=1, errors='ignore')
    X_test = test_df.drop(['id'], axis=1, errors='ignore')
    
    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['float64', 'int64', 'float32']).columns.tolist()
    
    X_native, X_test_native, X_nn, X_test_nn = prepare_data_dual(X, X_test, cat_cols, num_cols)
    print(f"Tree Data: {X_native.shape} | NN Data: {X_nn.shape}")
    
    native_cat_cols = X_native.select_dtypes(include=['category']).columns.tolist()

    # --- MODEL DEFINITIONS (7 Models) ---
    
    # 1. Main XGB (Tuned)
    xgb_params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'learning_rate': 0.0029, 
                  'max_depth': 5, 'subsample': 0.39, 'colsample_bytree': 0.32, 'n_estimators': 15000, 
                  'tree_method': 'gpu_hist', 'enable_categorical': True, 'grow_policy': 'lossguide', 
                  'alpha': 2.6, 'lambda': 0.69, 'min_child_weight': 5, 'max_bin': 512, 'random_state': 2025}
    
    # 2. Main LGBM (Robust)
    lgb_params = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.015, 'num_leaves': 90, 
                  'max_depth': 8, 'n_estimators': 5000, 'colsample_bytree': 0.7, 'subsample': 0.8, 
                  'device': 'gpu', 'verbose': -1, 'random_state': 2025}
    
    # 3. Main CatBoost
    cat_params = {'iterations': 5000, 'learning_rate': 0.015, 'depth': 7, 'l2_leaf_reg': 5.0, 
                  'task_type': 'GPU', 'eval_metric': 'AUC', 'verbose': False, 'random_seed': 2025}

    # 4. LGBM Extra Trees (Diversity)
    lgb_xt_params = {'objective': 'binary', 'metric': 'auc', 'learning_rate': 0.02, 'n_estimators': 5000,
                     'extra_trees': True, 'num_leaves': 64, 'max_depth': 7, 'device': 'gpu', 
                     'verbose': -1, 'random_state': 2025}

    # 5. CatBoost Shallow (Diversity)
    cat_shallow_params = {'iterations': 3000, 'learning_rate': 0.03, 'depth': 3, 'l2_leaf_reg': 3.0, 
                          'task_type': 'GPU', 'eval_metric': 'AUC', 'verbose': False, 'random_seed': 2025}

    # 6. XGBoost DART (Diversity)
    # DART is slow, so we reduce estimators slightly and use aggressive learning rate
    xgb_dart_params = {'objective': 'binary:logistic', 'eval_metric': 'auc', 'booster': 'dart', 
                       'rate_drop': 0.1, 'skip_drop': 0.5, 'learning_rate': 0.05, 'max_depth': 6, 
                       'n_estimators': 1500, 'tree_method': 'gpu_hist', 'enable_categorical': True, 
                       'random_state': 2025}

    models = {
        'xgb': xgb.XGBClassifier(**xgb_params),
        'lgb': lgb.LGBMClassifier(**lgb_params),
        'cat': CatBoostClassifier(**cat_params),
        'lgb_xt': lgb.LGBMClassifier(**lgb_xt_params),
        'cat_sh': CatBoostClassifier(**cat_shallow_params),
        'xgb_dart': xgb.XGBClassifier(**xgb_dart_params)
    }

    # --- Training Loop ---
    skf = StratifiedKFold(n_splits=Config.n_folds, shuffle=True, random_state=Config.seed)
    
    oof = {m: np.zeros(len(X)) for m in models.keys()}
    oof['nn'] = np.zeros(len(X))
    
    test_p = {m: np.zeros(len(X_test)) for m in models.keys()}
    test_p['nn'] = np.zeros(len(X_test))

    print(f"\n--- Starting {Config.n_folds}-Fold Massive Stacking ---")

    for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
        print(f"\n>>> Fold {fold+1}/{Config.n_folds}")
        
        # Slicing
        X_tr, y_tr = X_native.iloc[train_idx], y.iloc[train_idx]
        X_val, y_val = X_native.iloc[val_idx], y.iloc[val_idx]
        
        # Train Tree Models
        for name, model in models.items():
            # Special handling for CatBoost features
            if 'cat' in name:
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], cat_features=native_cat_cols, verbose=False, early_stopping_rounds=200)
            elif 'lgb' in name:
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], callbacks=[lgb.early_stopping(100, verbose=False)])
            else: # XGB
                model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False, early_stopping_rounds=200)
            
            p_val = model.predict_proba(X_val)[:, 1]
            oof[name][val_idx] = p_val
            test_p[name] += model.predict_proba(X_test_native)[:, 1] / Config.n_folds
            print(f"    {name.upper()}: {roc_auc_score(y_val, p_val)}")

        # Train NN
        keras.backend.clear_session()
        model_nn = get_nn_model(X_nn.shape[1])
        es = callbacks.EarlyStopping(monitor='val_auc', patience=7, mode='max', restore_best_weights=True)
        lr = callbacks.ReduceLROnPlateau(monitor='val_auc', factor=0.5, patience=3, verbose=0)
        
        model_nn.fit(X_nn[train_idx], y.iloc[train_idx], 
                     validation_data=(X_nn[val_idx], y.iloc[val_idx]), 
                     epochs=40, batch_size=512, callbacks=[es, lr], verbose=0)
        
        oof['nn'][val_idx] = model_nn.predict(X_nn[val_idx], verbose=0).flatten()
        test_p['nn'] += model_nn.predict(X_test_nn, verbose=0).flatten() / Config.n_folds
        print(f"    NN: {roc_auc_score(y.iloc[val_idx], oof['nn'][val_idx])}")
        
        gc.collect()

    # --- Submit ---
    oof_df = pd.DataFrame(oof)
    oof_df[Config.target] = y.values
    test_preds_df = pd.DataFrame(test_p)

    final_preds, final_auc = hill_climbing(oof_df, Config.target, test_preds_df)

    sub_filename = f"submission_7ModelStack_AUC_{final_auc}.csv"
    pd.DataFrame({'id': test_ids, Config.target: final_preds}).to_csv(sub_filename, index=False)
    print(f"\nSUCCESS! Submission saved to: {sub_filename}")

2025-12-08 15:26:14.230546: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1765207574.430183      47 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1765207574.481907      47 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

...

    CAT_SH: 0.728840574793467
    XGB_DART: 0.7287541419422556
    NN: 0.7218621026051133

>>> Fold 3/10
    XGB: 0.7291527366164023
    LGB: 0.7293853073613128


Default metric period is 5 because AUC is/are not implemented for GPU


    CAT: 0.7299410968048117
    LGB_XT: 0.7265095974884487


Default metric period is 5 because AUC is/are not implemented for GPU


    CAT_SH: 0.7287452045691009


In [ ]:
# max auc : 0.7308856727681804